# Implementing control variates yourself

A hands-on companion to `control_variates.ipynb`. You write the functions; the cell after each exercise checks
them. If you get stuck, set `USE_SOLUTIONS = True` below (the reference solutions are in `cv_solutions.py`, next to
this notebook), but try first.

1. A scalar warm-up: the estimator and the optimal $\beta$
2. Where $\beta$ comes from when you have only one simulation
3. Power spectra of a field, with an exactly known control mean
4. Why the Zel'dovich field and not linear theory, and the one big pitfall
5. The same on FML output files: hybrid bias spectra and MG/GR boosts

Everything runs in seconds.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

USE_SOLUTIONS = False   # True: use cv_solutions.py wherever an exercise is not done
rng = np.random.default_rng(42)

## 1. Warm-up: estimating a mean

We want $\mu_X = \langle X\rangle$ with $X = e^{Z/2}$, $Z \sim \mathcal N(0,1)$ (the true answer is $e^{1/8}$).
As a control we use $Y = Z$ itself: computed from the same random numbers, and its mean is known exactly,
$\langle Y\rangle = 0$.

**Exercise 1.** Write `cv_estimate(x, y, mu_y, beta)` returning $x - \beta\,(y - \mu_y)$ (elementwise on arrays).

**Exercise 2.** Write `optimal_beta(x, y)` returning $\beta^* = \mathrm{Cov}(x,y)/\mathrm{Var}(y)$ from samples along
the first axis (it should work for 1D arrays and for arrays of shape `[n_samples, n_k]`).

In [ ]:
def cv_estimate(x, y, mu_y, beta):
    # YOUR CODE HERE
    raise NotImplementedError

def optimal_beta(x, y):
    # YOUR CODE HERE
    raise NotImplementedError

if USE_SOLUTIONS:
    from cv_solutions import cv_estimate, optimal_beta

In [ ]:
# Check
z = rng.standard_normal(100000)
x, y = np.exp(z / 2), z
assert np.allclose(cv_estimate(np.array([3.0]), np.array([1.0]), 0.5, 2.0), [2.0])
b = optimal_beta(x, y)
assert abs(b - 0.5 * np.exp(1 / 8)) < 0.02, b   # Cov(e^{Z/2}, Z) = e^{1/8} / 2
assert np.allclose(optimal_beta(np.c_[x, 2 * x], np.c_[y, y]), [b, 2 * b])
print("Exercises 1-2 OK; beta* =", round(b, 3))

Now the payoff. Repeat an "experiment" of $n = 10$ samples 5000 times, and compare the scatter of the plain mean
with that of the control-variate mean for several $\beta$. Note that **every** $\beta$ gives the right average
(unbiased); $\beta$ only changes the scatter, and $\beta^*$ minimises it, reducing the variance by $1/(1-\rho^2)$.

In [ ]:
n, nexp = 10, 5000
Z = rng.standard_normal((nexp, n))
X, Y = np.exp(Z / 2), Z
rho = np.corrcoef(X.ravel(), Y.ravel())[0, 1]
print(f"correlation rho = {rho:.3f}: predicted variance reduction with beta* = {1 / (1 - rho**2):.1f}x")
plt.figure(figsize=(6.5, 3.5))
for beta in [0.0, 0.3, b, 1.2]:
    est = cv_estimate(X, Y, 0.0, beta).mean(1)
    lab = f"beta = {beta:.2f}: mean {est.mean():.4f}, var reduction {X.mean(1).var() / est.var():.1f}x"
    plt.hist(est, bins=60, histtype="step", label=lab)
plt.axvline(np.exp(1 / 8), color="k", label="truth")
plt.legend(fontsize=7); plt.xlabel("estimate of <X> from 10 samples"); plt.tight_layout()

## 2. Where does $\beta$ come from?

In section 1 we estimated $\beta^*$ from 100 000 samples. In cosmology you have **one** simulation per cosmology,
and you cannot estimate a covariance from one sample. The answer used in practice (and in FML):

* fix $\beta(k)$ **in advance**: $\beta = 1$ where the simulation and the control are nearly the same field (large
  scales), going smoothly to 0 where they decorrelate, with the filter of Kokron et al. (2022):
  $\beta(k) = \tfrac12\left[1 - \tanh\left((k-k_0)/\Delta k\right)\right]$, $k_0 = 0.618$, $\Delta k = 0.167$ h/Mpc;
* any fixed $\beta(k)$ keeps the estimate unbiased; a bad choice only costs variance.

Estimating $\beta$ from a few samples *and* using the same samples also has a subtle cost: the estimate is no longer
exactly unbiased: the error is of order $1/n$, and with few samples it is large (the cell after the exercise
finds about 5% with $n = 5$, far more than the statistical error). Another reason to fix $\beta$ in advance.

**Exercise 3.** Write `beta_tanh(k, k0=0.618, dk=0.167)`.

In [ ]:
def beta_tanh(k, k0=0.618, dk=0.167):
    # YOUR CODE HERE
    raise NotImplementedError

if USE_SOLUTIONS:
    from cv_solutions import beta_tanh

In [ ]:
# Check
assert np.isclose(beta_tanh(0.618), 0.5) and beta_tanh(0.0) > 0.99 and beta_tanh(2.0) < 0.001
print("Exercise 3 OK")

# In-sample beta from n = 5 samples: a small bias (compare with the fixed beta*)
n = 5
Z = rng.standard_normal((200000, n)); X, Y = np.exp(Z / 2), Z
b_in = np.array([optimal_beta(X[i], Y[i]) for i in range(20000)])
est_in = (X[:20000] - b_in[:, None] * Y[:20000]).mean(1)
est_fix = cv_estimate(X, Y, 0.0, b).mean(1)
print(f"truth {np.exp(1/8):.4f} | fixed beta: {est_fix.mean():.4f} +- {est_fix.std() / np.sqrt(len(est_fix)):.4f}"
      f" | beta from the same 5 samples: {est_in.mean():.4f} +- {est_in.std() / np.sqrt(len(est_in)):.4f}")

## 3. Power spectra with an exactly known control mean

Now a field. On a 2D grid we draw a Gaussian linear field $\delta_L$ and make a "simulation" by a nonlinear local
transform, $\delta_\mathrm{sim} = e^{\delta_L - \sigma^2/2} - 1$ (a lognormal field: it couples the modes like
gravity does, roughly). The control is the power spectrum of $\delta_L$ itself, whose mean is known **exactly**: the
input $P_\mathrm{lin}(k)$.

The helpers below (grid, field, binned power spectrum) are given.

In [ ]:
N, L = 128, 256.0
kx = np.fft.fftfreq(N, d=L / N) * 2 * np.pi; ky = np.fft.rfftfreq(N, d=L / N) * 2 * np.pi
KX, KY = np.meshgrid(kx, ky, indexing="ij"); K = np.sqrt(KX**2 + KY**2)
nbins = 25
edges = np.linspace(np.pi / L, np.pi * N / L, nbins + 1); kbin = 0.5 * (edges[1:] + edges[:-1])
which = np.digitize(K.ravel(), edges) - 1
weight = np.ones_like(K); weight[:, 1:-1] = 2; weight = weight.ravel()   # rfft: ky > 0 counts twice
ok = (which >= 0) & (which < nbins)
nmodes = np.bincount(which[ok], weights=weight[ok], minlength=nbins)

def P_lin(k):
    return 300.0 * (k / 0.05) / (1 + (k / 0.05)**2.5) * np.exp(-(k / 0.8)**2)

def linear_field(seed, fixed=False):
    w = np.fft.rfft2(np.random.default_rng(seed).standard_normal((N, N)))
    if fixed:
        w = w / np.abs(w) * N
    d = w * np.sqrt(P_lin(np.where(K > 0, K, 1.0)) / L**2) * N
    d[0, 0] = 0
    return np.fft.irfft2(d, s=(N, N))

def power(delta):
    p = (np.abs(np.fft.rfft2(delta))**2).ravel() * weight * L**2 / N**4
    return np.bincount(which[ok], weights=p[ok], minlength=nbins) / np.maximum(nmodes, 1)

def simulation(dL):
    return np.exp(dL - dL.var() / 2) - 1

**Exercise 4.** The control mean must be computed **exactly like the measurement**: here, the average of
$P_\mathrm{lin}(|k|)$ over the modes in each bin, with the same weights, not $P_\mathrm{lin}$ at the bin centre (the
difference is large in the lowest bins). Write `binned_mean(P_of_k, K, which, weight, nbins)`.

**Exercise 5.** Write `variance_reduction(P_sims, P_cvs)` = Var/Var per $k$ bin over realisations (first axis).

In [ ]:
def binned_mean(P_of_k, K, which, weight, nbins):
    # YOUR CODE HERE
    raise NotImplementedError

def variance_reduction(P_sims, P_cvs):
    # YOUR CODE HERE
    raise NotImplementedError

if USE_SOLUTIONS:
    from cv_solutions import binned_mean, variance_reduction

In [ ]:
# Check: the binned mean is the average power of many linear fields
C_mean = binned_mean(P_lin, K, which, weight, nbins)
avg = np.mean([power(linear_field(s)) for s in range(400)], axis=0)
assert np.all(np.abs(avg / C_mean - 1) < 4 / np.sqrt(400 * nmodes) + 0.01), (avg / C_mean)[:5]
assert np.allclose(variance_reduction(np.array([[1.0], [3.0]]), np.array([[1.0], [2.0]])), [4.0])
print("Exercises 4-5 OK; lowest bin: P_lin(centre) / binned mean =", round(P_lin(kbin[0]) / C_mean[0], 3))

**Your turn.** Fill in the loop: for each seed, the simulation's spectrum `P_sim`, the control `P_ctl`, and the
control-variate estimate with $\beta = 1$. Then plot the variance reduction. (A solution is in the next cell, commented
out.)

In [ ]:
seeds = range(200)
P_sim, P_ctl, P_cv = [], [], []
for s in seeds:
    dL = linear_field(s)
    # YOUR CODE HERE: append power(simulation(dL)), power(dL), and the control-variate estimate
    pass
# --- solution ---
# for s in seeds:
#     dL = linear_field(s)
#     P_sim.append(power(simulation(dL))); P_ctl.append(power(dL))
#     P_cv.append(cv_estimate(P_sim[-1], P_ctl[-1], C_mean, 1.0))
if USE_SOLUTIONS and not P_sim:
    for s in seeds:
        dL = linear_field(s)
        P_sim.append(power(simulation(dL))); P_ctl.append(power(dL)); P_cv.append(cv_estimate(P_sim[-1], P_ctl[-1], C_mean, 1.0))
P_sim, P_ctl, P_cv = map(np.array, (P_sim, P_ctl, P_cv))

beta_star = optimal_beta(P_sim, P_ctl)
P_cv_opt = cv_estimate(P_sim, P_ctl, C_mean, beta_star)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].plot(kbin, variance_reduction(P_sim, P_cv), label="beta = 1")
ax[0].plot(kbin, variance_reduction(P_sim, P_cv_opt), "--", label="beta* (estimated from the 200 seeds)")
ax[0].axhline(1, color="k"); ax[0].set(xscale="log", yscale="log", xlabel="k", ylabel="variance reduction"); ax[0].legend()
ax[1].plot(kbin, beta_star); ax[1].set(xscale="log", xlabel="k", ylabel="beta*(k)")
plt.tight_layout()

**Questions to think about.** Why is $\beta^*$ close to 1 at low $k$? Why does the gain drop at high $k$? (The
lognormal transform moves power between scales; at high $k$ the simulation's power comes from couplings of many
modes, which the linear power of the *same* bin does not know about.)

## 4. Fixed amplitudes, the Zel'dovich field, and the pitfall

**Try this:** rerun the loop above with `linear_field(s, fixed=True)` (amplitude-fixed initial conditions, as your
runs). `P_ctl` then has **no variance at all** ($|\delta_k|^2 = P$ exactly), so it can't remove anything. Yet the
simulation still scatters, through the phases. That is why FML uses the **Zel'dovich field** as the control: it is
nonlinear in the initial field (particles moved by the 1LPT displacement), so it scatters with the phases in the same
way as the simulation, and its mean is still known analytically (the Zel'dovich prediction).

Below, a small particle version: the "simulation" moves the lattice with 2LPT, the control with Zel'dovich (1LPT),
both from fixed-amplitude initial conditions. In FML $\langle C\rangle$ is the analytic prediction; here we stand in
for it with the mean of 200 *other* Zel'dovich realisations.

In [ ]:
q = (np.indices((N, N)) * (L / N)).reshape(2, -1)
K2 = np.where(K > 0, K**2, 1.0)

def lpt_fields(seed, fixed=True):
    d = np.fft.rfft2(linear_field(seed, fixed))
    phi = -d / K2
    psi1 = [np.fft.irfft2(-1j * kk * phi, s=(N, N)) for kk in (KX, KY)]
    pxx, pyy, pxy = (np.fft.irfft2(-a * b * phi, s=(N, N)) for a, b in ((KX, KX), (KY, KY), (KX, KY)))
    phi2 = -np.fft.rfft2(pxx * pyy - pxy**2) / K2
    psi2 = [-3 / 7 * np.fft.irfft2(-1j * kk * phi2, s=(N, N)) for kk in (KX, KY)]
    return psi1, [a + b for a, b in zip(psi1, psi2)]

def paint(psi):
    x = (q + np.array([p.ravel() for p in psi])) / (L / N) % N
    i = np.floor(x).astype(int); f = x - i; rho = np.zeros(N * N)
    for dx in (0, 1):
        for dy in (0, 1):
            np.add.at(rho, ((i[0] + dx) % N) * N + (i[1] + dy) % N, (f[0] if dx else 1 - f[0]) * (f[1] if dy else 1 - f[1]))
    return rho.reshape(N, N) - 1

def zel_and_sim(seed):
    psi_z, psi_s = lpt_fields(seed)
    return power(paint(psi_s)), power(paint(psi_z))

runs = np.array([zel_and_sim(s) for s in range(100)])          # [seed, sim/zel, k]
C_zel = np.mean([zel_and_sim(1000 + s)[1] for s in range(200)], axis=0)   # stand-in for the analytic prediction
Ps, Pz = runs[:, 0], runs[:, 1]
print("variance reduction at the lowest k bins (beta = 1):", variance_reduction(Ps, cv_estimate(Ps, Pz, C_zel, 1.0))[:4].round(1))

**The pitfall.** Suppose you didn't have an analytic $\langle C\rangle$ and used the mean of the *same* runs
instead. Average the estimate over the runs: $\overline{P - (C - \bar C)} = \bar P$. You've gained nothing, although
each run's estimate looks beautifully smooth. And a $\langle C\rangle$ that is slightly off (a different grid, cut or
binning than the measurement) puts its error straight into the result, however many runs you average. Check both:

In [ ]:
same = cv_estimate(Ps, Pz, Pz.mean(0), 1.0)
print("ensemble mean with <C> from the same runs equals the plain mean:", np.allclose(same.mean(0), Ps.mean(0)))
off = cv_estimate(Ps, Pz, 1.01 * C_zel, 1.0).mean(0) / cv_estimate(Ps, Pz, C_zel, 1.0).mean(0) - 1
print("a 1% error in <C> gives a bias of", (100 * off[:3]).round(2), "% (lowest bins)")

## 5. On FML output

A FML run with

```lua
hybridbias_kcut_over_nyquist = 1.0      -- any value > 0, the same in every run you compare
hybridbias_zeldovich_field = true       -- the control, in every run (~80 s per output at full size)
hybridbias_zeldovich = true             -- <C>: once per cosmology (and gravity model)
```

writes, in `snapshot_*_z<z>/`, `hybridbias_z<z>.txt` (the spectra), `hybridbias_zeldovichfield_z<z>.txt` (the
control) and `hybridbias_zeldovich_z<z>.txt` (its mean). Each file has the columns k, number of modes, the 15
spectra in the order $(i,j)$, $i \le j$ of the operators $(1, \delta, \delta^2, s^2, \nabla^2\delta)$, and then
$P_\mathrm{cb,lin}$.

**Exercise 7.** Write

* `load_hybridbias(path)` returning `k, nmodes, P` with `P` of shape `[15, nk]`;
* `control_variate(P, P_field, P_pred, k, k0=0.618, dk=0.167)` for all 15 spectra at once;
* `boost_cv(P_mg, Pf_mg, Ppred_mg, P_gr, Pf_gr, Ppred_gr, k)`: the MG/GR boost with the control variate applied to
  each model separately, each with its **own** prediction (the Zel'dovich fields of MG and GR differ through their
  growth factors).

(Exercise 6 was the loop in section 3.) The check builds fake files in the FML format.

In [ ]:
def load_hybridbias(path):
    # YOUR CODE HERE
    raise NotImplementedError

def control_variate(P, P_field, P_pred, k, k0=0.618, dk=0.167):
    # YOUR CODE HERE
    raise NotImplementedError

def boost_cv(P_mg, Pf_mg, Ppred_mg, P_gr, Pf_gr, Ppred_gr, k, k0=0.618, dk=0.167):
    # YOUR CODE HERE
    raise NotImplementedError

if USE_SOLUTIONS:
    from cv_solutions import load_hybridbias, control_variate, boost_cv

In [ ]:
# Check with fake files in the FML format
import os, tempfile
tmp = tempfile.mkdtemp()
k = np.linspace(0.01, 1.5, 50); nm = np.arange(50) * 10.0 + 4
truth = np.array([1000 / (1 + (k / 0.2)**2) * (1 + 0.1 * a) for a in range(15)])
noise = 1 + 0.05 * np.sin(13 * k)                       # the same "unlucky modes" in the run and its control
def write(name, P):
    np.savetxt(os.path.join(tmp, name), np.column_stack([k, nm, P.T, truth[0]]), header="fake FML file")
write("run.txt", truth * noise); write("field.txt", 0.8 * truth * noise); write("pred.txt", 0.8 * truth)
kk, nmodes_read, P = load_hybridbias(os.path.join(tmp, "run.txt"))
assert P.shape == (15, 50) and np.allclose(kk, k) and np.allclose(nmodes_read, nm)
Pf, Pp = load_hybridbias(os.path.join(tmp, "field.txt"))[2], load_hybridbias(os.path.join(tmp, "pred.txt"))[2]
Pcv = control_variate(P, Pf, Pp, k)
low = k < 0.3
assert np.all(np.abs(Pcv[:, low] / truth[:, low] - 1) < np.abs(P[:, low] / truth[:, low] - 1) + 1e-12)
assert np.allclose(boost_cv(P, Pf, Pp, P, Pf, Pp, k), 1.0)
print("Exercise 7 OK: the control variate removes the fake noise on large scales")
plt.figure(figsize=(6, 3.5)); plt.plot(k, P[3] / truth[3], label="run"); plt.plot(k, Pcv[3] / truth[3], label="with control variate")
plt.axhline(1, color="k"); plt.xlabel("k (h/Mpc)"); plt.ylabel("P_03 / truth"); plt.legend(); plt.tight_layout()

### Using it on your runs

```python
import glob
run = "../results/scans/<scan>/<point>_seed1001_normal/snapshot_TestSim_z0.000/"
k, nm, P = load_hybridbias(run + "hybridbias_z0.000.txt")
Pf = load_hybridbias(run + "hybridbias_zeldovichfield_z0.000.txt")[2]
Pp = load_hybridbias("<the run of this cosmology with hybridbias_zeldovich = true>/hybridbias_zeldovich_z0.000.txt")[2]
Pcv = control_variate(P, Pf, Pp, k)
```

The prediction has to come from a run with the **same** cosmology, gravity model, redshift, `hybridbias_nmesh`,
`hybridbias_kcut_over_nyquist` and smoothing; the seed doesn't matter.

**Further things to try**
* In section 4, replace $\beta = 1$ by `beta_tanh(kbin, k0, dk)` with different `k0`; where does the gain turn into a
  loss?
* Make the field more nonlinear (the amplitude in `P_lin`) and watch the correlation drop.
* Estimate $\beta^*$ from only 5 seeds in section 3 and apply it to the others: how much is lost?
* Use `cv_estimate` on the MG/GR boost of the particle toy: give `lpt_fields` a growth factor that differs between two
  "models", compute both, and compare the scatter of the boost with and without the control variate.